# ASMSA v2 &mdash; Analysis

&#8592; [Metadynamics](06-md.ipynb)

| | |
|---|---|
| **What it does** | Read the latent space as structural biology: find the states, export representative structures, characterize them, explore interactively. |
| **Typical runtime** | a few minutes |
| **Needs** | a checkpoint and latent coordinates from <a href='04-train.ipynb'>04-train</a> |
| **Produces** | cluster PDBs, per-state statistics, interactive plots |

---

Up to here the latent space has been a scatter plot. This notebook turns it into
statements about the system: *how many states are there, which structures are they,
what distinguishes them, and how does the system move between them.*

It works on the training trajectory, and equally on a metadynamics trajectory from
[06-md](06-md.ipynb) &mdash; see the switch in section 1.

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import mdtraj as md

import asmsa_v2 as asmsa
from asmsa_v2.viz import use_asmsa_style

use_asmsa_style()
project = asmsa.Project.load('run')
asmsa.set_seed(0)

## 1. Which trajectory to analyse

`'training'` uses the input trajectory and its saved latent. `'md'` uses the
metadynamics trajectory from [06-md](06-md.ipynb), if it exists &mdash; the interesting
case, since that is where the new sampling is.

In [2]:
SOURCE = 'md'       # 'training' | 'md'

md_latent = project.dir('md') / 'latent-md.npy'
train_latent = project.dir('models') / f'latent-{project.model_name}.npy'

if SOURCE == 'md' and md_latent.exists():
    z = np.load(md_latent)
    traj = md.load(str(project.dir('md') / 'fit.xtc'),
                   top=str(project.dir('md') / f'{project.name}-box.gro'))
    traj = traj.atom_slice(traj.top.select('element != H'))
    label = 'metadynamics'
else:
    if SOURCE == 'md':
        print('no metadynamics latent found; falling back to the training trajectory')
    z = np.load(train_latent)
    traj = md.load(str(project.traj_path), top=str(project.conf_path))
    align = traj.top.select('name CA')
    if len(align) < 3:
        align = traj.top.select('element != H')
    traj.superpose(traj, frame=0, atom_indices=align)
    label = 'training'

assert len(z) == traj.n_frames, (
    f'{len(z)} latent points but {traj.n_frames} frames -- '
    f'the latent and the trajectory must correspond frame for frame')

components = {k: traj.top.select(v) for k, v in (project.components or {}).items()}
print(f'analysing the {label} trajectory: {traj.n_frames} frames, '
      f'{traj.n_atoms} atoms, latent_dim={z.shape[1]}')

no metadynamics latent found; falling back to the training trajectory
analysing the training trajectory: 50001 frames, 144 atoms, latent_dim=2


## 2. The landscape

**What to look for:** distinct dark basins. Each is a candidate state. The trajectory
panel shows whether the system actually travels between them (useful) or sits in one
(you have one state, or insufficient sampling).

In [ ]:
props = asmsa.viz.standard_properties(traj, components or None)
fig = asmsa.viz.latent_overview(z, properties=props, in_time_order=True)
plt.show()

## 3. Find the states

HDBSCAN is density-based: it does not need to be told how many clusters there are, and
it marks sparsely populated frames as noise (`-1`) rather than forcing them into the
nearest basin. `min_cluster_size` is the one knob &mdash; it is the smallest number of
frames you are willing to call a state. Raise it to merge marginal clusters, lower it to
split.

If HDBSCAN is not installed, KMeans is used and `k` is chosen by silhouette score.

**What to look for:** a cluster count that matches the number of basins you can see, and
a noise fraction well under half. Mostly-noise means `min_cluster_size` is too large, or
the latent has no real structure.

In [ ]:
MIN_CLUSTER_SIZE = 400 #max(20, len(z) // 40)

result = asmsa.cluster_latent(z, min_cluster_size=MIN_CLUSTER_SIZE)
print(result.summary())

noise_frac = float((result.labels == -1).mean())
print(f'\nnoise fraction: {noise_frac:.1%}', end='  ')
print('(fine)' if noise_frac < 0.4 else '(high -- try a smaller MIN_CLUSTER_SIZE)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

asmsa.viz.latent_colored_by(z, result.labels, ax=axes[0], cmap='tab10',
                            cbar_label='cluster id (-1 = noise)',
                            title=f'{result.method}', s=5, alpha=0.6)
for cid, centre in result.centroids.items():
    axes[0].scatter(*centre, marker='*', s=260, c='k', edgecolors='w', linewidths=1.5)
    axes[0].annotate(str(cid), centre, textcoords='offset points', xytext=(9, 6),
                     fontweight='bold')

asmsa.viz.free_energy_surface(z, ax=axes[1], max_FE=8.0)
for cid, centre in result.centroids.items():
    axes[1].scatter(*centre, marker='*', s=260, c='w', edgecolors='k', linewidths=1.2)
    axes[1].annotate(str(cid), centre, textcoords='offset points', xytext=(9, 6),
                     color='w', fontweight='bold')

for a in axes:
    asmsa.viz.latent_axes(a)
plt.tight_layout(); plt.show()

## 4. Export a structure per state

The representative is the frame closest to its cluster's centroid in latent space.
These PDBs are the deliverable: open them in PyMOL or ChimeraX and look at what
distinguishes the states.

For a protein-ligand complex, the thing to check is whether the ligand actually sits in
different places &mdash; that is what it means for the latent to have resolved the
binding poses.

In [ ]:
pose_dir = project.dir('analysis') / f'states-{label}'
paths = asmsa.save_cluster_representatives(result, traj, pose_dir,
                                           prefix='state')
for cid, p in sorted(paths.items()):
    print(f'  state {cid}: {result.sizes[cid]:>5} frames  ->  '
          f'{p.relative_to(project.root)}')

### What distinguishes the states?

Per-state means of the structural observables. This is where a cluster becomes
interpretable: *state 0 has the ligand 1.3 nm from the protein centre, state 1 has it
at 1.5 nm*, and so on.

**What to look for:** at least one observable that clearly separates the states. If
every column is the same across states, the clusters are a partition of noise rather
than real structural distinctions.

In [ ]:
try:
    import pandas as pd
    rows = []
    for cid in result.cluster_ids:
        mask = result.labels == cid
        row = {'state': cid, 'frames': int(mask.sum()),
               'share': f'{mask.mean():.1%}'}
        for name, vals in props.items():
            if name == 'frame':
                continue
            row[name] = float(np.mean(np.asarray(vals)[mask]))
        rows.append(row)
    table = pd.DataFrame(rows).set_index('state')
    display(table.round(3))
except ImportError:
    for cid in result.cluster_ids:
        mask = result.labels == cid
        print(f'state {cid}: {mask.sum()} frames')
        for name, vals in props.items():
            if name != 'frame':
                print(f'    {name:<34} {np.mean(np.asarray(vals)[mask]):.3f}')

In [ ]:
# Distributions, not just means -- overlapping distributions mean the states are
# not really distinguished by that observable, whatever the means say.
plot_props = [k for k in props if k != 'frame']
if plot_props and result.cluster_ids:
    fig, axes = plt.subplots(1, len(plot_props),
                             figsize=(3.6 * len(plot_props), 2.9), squeeze=False)
    for ax, name in zip(axes[0], plot_props):
        vals = np.asarray(props[name])
        for cid in result.cluster_ids:
            ax.hist(vals[result.labels == cid], bins=30, alpha=0.55,
                    label=f'state {cid}')
        ax.set_xlabel(name, fontsize=8)
        ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

## 5. How does the system move between states?

Only meaningful for a trajectory in time order.

The transition count matrix shows which states interconvert directly. Diagonal
dominance is expected (frames are correlated); what matters are the **off-diagonal**
entries. A state with no off-diagonal counts was entered or left at most once, so its
population is not statistically meaningful however many frames it contains.

In [ ]:
labels = result.labels
ids = result.cluster_ids
if len(ids) >= 2:
    index = {cid: i for i, cid in enumerate(ids)}
    counts = np.zeros((len(ids), len(ids)), dtype=int)
    prev = None
    for lab in labels:
        if lab < 0:
            continue
        if prev is not None:
            counts[index[prev], index[lab]] += 1
        prev = lab

    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    im = ax.imshow(np.log10(counts + 1), cmap='viridis')
    ax.set_xticks(range(len(ids)), ids); ax.set_yticks(range(len(ids)), ids)
    ax.set_xlabel('to state'); ax.set_ylabel('from state')
    ax.set_title('transitions (log$_{10}$ count)')
    for i in range(len(ids)):
        for j in range(len(ids)):
            ax.text(j, i, counts[i, j], ha='center', va='center',
                     color='w', fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.045)
    ax.grid(False)
    plt.tight_layout(); plt.show()

    off = counts.sum() - np.trace(counts)
    print(f'{off} inter-state transitions observed')
    if off < 10:
        print('  -> too few to treat the state populations as converged')
else:
    print('fewer than two states; nothing to transition between')

### Are the CVs slow?

Repeated here because it is the question that decides whether these states are kinetic
states or just density features. **What to look for:** implied timescales that rise and
then plateau; only the plateau value means anything.

In [ ]:
asmsa.viz.vamp_scan_plot(z, dt=1.0)
plt.tight_layout(); plt.show()

report = asmsa.evaluate_latent(z, prior=project.make_prior(), vamp_lag=20)
report

## 6. Interactive exploration

Hover a point to see its frame index. Requires `plotly`.

In [ ]:
try:
    from asmsa_v2.viz.interactive import interactive_latent
    hover = [f'frame {i} | state {result.labels[i]}' for i in range(len(z))]
    fig = interactive_latent(z, labels=result.labels, hover_text=hover,
                             colorbar_label='state',
                             title=f'latent space ({label}) -- hover for frame')
    fig.show()
except ImportError as exc:
    print('interactive plot unavailable:', exc)
    print('install it with:  pip install plotly')

### Click a point, see the structure

Links the latent scatter to a 3-D viewer: clicking a point renders that frame. The most
direct way to build intuition for what a region of the latent space *means*.

Requires `plotly`, `py3Dmol` and `ipywidgets`, and a live kernel &mdash; it will not
render in a statically exported notebook.

In [ ]:
try:
    from asmsa_v2.viz.interactive import latent_browser
    from IPython.display import display
    palette = ['cyan', 'magenta', 'yellow', 'orange']
    browser = latent_browser(
        z, traj,
        labels=result.labels,
        component_atoms=components or None,
        component_colors=({k: c for k, c in zip(components, palette)}
                          if components else None),
        structure_style='sticks',
    )
    display(browser)
except ImportError as exc:
    print('structure browser unavailable:', exc)
    print('install it with:  pip install plotly py3Dmol ipywidgets anywidget')

## 7. Decode a latent point back to a structure

The decoder maps any point in the latent plane back to internal coordinates &mdash;
including points no frame ever visited, which is how you ask *what would a structure
halfway between these two states look like.*

An important caveat: the decoder returns **internal coordinates**, not Cartesian
positions. Reconstructing 3-D coordinates from them is a separate problem (distance-
geometry embedding) and is not part of this pipeline. What the decoded values are good
for is reading off *which* internal coordinates change between states, and sanity-
checking that the decoder has learned physically plausible geometry &mdash; bond lengths
coming back near 0.15 nm, for instance.

In [ ]:
decoder_path = project.dir('models') / f'{project.cv_stem}-decoder.pt'
scaler = asmsa.FeatureScaler.load(project.scaler_file)
bundle = torch.load(project.features_file, weights_only=False)
blocks = bundle['feature_blocks']

if decoder_path.exists() and len(result.cluster_ids) >= 2:
    decoder = torch.jit.load(str(decoder_path)); decoder.eval()
    a, b = result.cluster_ids[0], result.cluster_ids[1]
    path = np.linspace(result.centroids[a], result.centroids[b], 9).astype(np.float32)
    with torch.no_grad():
        decoded_std = decoder(torch.from_numpy(path)).numpy()
    decoded = scaler.inverse_transform(decoded_std)

    print(f'interpolating from state {a} to state {b}, in physical units:')
    for name, (lo, hi) in blocks.items():
        seg = decoded[:, lo:hi]
        print(f'  {name:<18} start {seg[0].mean():+.3f}  end {seg[-1].mean():+.3f}  '
              f'(mean over {hi - lo} features)')

    changed = np.abs(decoded[-1] - decoded[0])
    top = np.argsort(changed)[::-1][:10]
    print(f'\nthe 10 internal coordinates that change most between the two states:')
    for i in top:
        block = next((n for n, (lo, hi) in blocks.items() if lo <= i < hi), '?')
        print(f'  feature {i:>5} ({block:<18}) '
              f'{decoded[0, i]:+.3f} -> {decoded[-1, i]:+.3f}  '
              f'(delta {changed[i]:+.3f})')
else:
    print('needs a decoder from 05-plumed and at least two states')

In [ ]:
if decoder_path.exists() and len(result.cluster_ids) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].scatter(z[:, 0], z[:, 1], s=2, alpha=0.15, color='0.6')
    axes[0].plot(path[:, 0], path[:, 1], 'o-', color='crimson', lw=1.6, ms=5)
    axes[0].set_title(f'interpolation path, state {a} -> {b}')
    asmsa.viz.latent_axes(axes[0])

    for name, (lo, hi) in blocks.items():
        axes[1].plot(decoded[:, lo:hi].mean(axis=1), 'o-', label=name, ms=4)
    axes[1].set_xlabel('step along the path')
    axes[1].set_ylabel('mean decoded value (physical units)')
    axes[1].set_title('how the internal coordinates change')
    axes[1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

## Summary

What you now have:

- a 2-D latent space whose basins are structural states, validated against held-out
  frames and scored for reconstruction, prior match, slow-dynamics content and spread;
- a representative PDB per state, with the observables that distinguish them;
- a TorchScript collective variable and a `plumed.dat` that bias along it;
- free energies from the biased run, by two independent estimators.

Where to go next:

| If | Then |
|---|---|
| the VAMP-2 score is low | set `time_lag` in [01-setup](01-setup.ipynb) and retrain &mdash; the single most effective change for CV quality |
| the states look unconvincing | revisit `inter_strategy` in [02-prepare](02-prepare.ipynb): if the inter-component features miss the binding site, the pose cannot be encoded |
| the metadynamics has not converged | extend the run in [06-md](06-md.ipynb) and re-check both convergence curves |
| you want a universal latent space across several complexes | pass more than one entry to `feature_dims`; the discriminator is molecule-agnostic, so the latent is shared |

---

&#8592; [Metadynamics](06-md.ipynb)